# Exploratory Data Analysis (EDA) for Artworks Dataset

This notebook performs Exploratory Data Analysis (EDA) on the Best Artworks of All Time dataset

The goal is to explore the dataset before using it for training a generative AI model that creates artworks inspired by famous artists.

In [ ]:

!pip install pandas numpy matplotlib seaborn pillow
!pip install kagglehub

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

## Download Dataset

In [ ]:
import kagglehub

path = kagglehub.dataset_download("ikarus777/best-artworks-of-all-time")

print("Path to dataset files:", path)

## Explore Dataset Structure

In [ ]:
import os

os.listdir(path)

## Access Resized Images Folder

In [ ]:
resized_path = os.path.join(path, "resized")

os.listdir(resized_path)

## Locate Image Files

In [ ]:
resized_path = os.path.join(path, "resized", "resized")

os.listdir(resized_path)

## Extract Artist Names from Image Files

In [ ]:
artists = os.listdir(resized_path)

print("Number of artists:", len(artists))
artists[:10]

## Image Distribution Across Artists

In [ ]:
artist_counts = {}

for img in artists:

    artist_name = "_".join(img.split("_")[:-1])

    if artist_name in artist_counts:
        artist_counts[artist_name] += 1
    else:
        artist_counts[artist_name] = 1

artist_counts

## Convert Artist Counts to DataFrame

In [ ]:
df = pd.DataFrame(list(artist_counts.items()), columns=["Artist", "Number_of_Images"])

df.head()

## Sort Artists by Number of Images

In [ ]:
df = df.sort_values(by="Number_of_Images", ascending=False)

df.head(10)

## Top 10 Artists by Number of Images

In [ ]:
plt.figure(figsize=(12,6))

sns.barplot(
    x="Number_of_Images",
    y="Artist",
    data=df.head(10)
)

plt.title("Top 10 Artists by Number of Images")
plt.show()

## Sample Images from the Dataset

In [ ]:
import random

plt.figure(figsize=(12,12))

for i in range(9):

    img_name = random.choice(artists)

    img_path = os.path.join(resized_path, img_name)

    img = Image.open(img_path)

    plt.subplot(3,3,i+1)
    plt.imshow(img)
    plt.title(img_name.split("_")[0])
    plt.axis("off")

plt.show()

In [ ]:
artist_name = "Vincent_van_Gogh"

artist_images = [img for img in artists if artist_name in img]

plt.figure(figsize=(10,10))

for i in range(9):
    img_name = random.choice(artist_images)
    img = Image.open(os.path.join(resized_path, img_name))

    plt.subplot(3,3,i+1)
    plt.imshow(img)
    plt.axis("off")

plt.suptitle(f"Sample Images of {artist_name}")
plt.show()

## Extract Image Dimensions

In [ ]:
sizes = []

for img_name in artists[:100]:

    img_path = os.path.join(resized_path, img_name)

    img = Image.open(img_path)

    sizes.append(img.size)

sizes[:10]

## Image Width Distribution

## Image Size Distribution

In [ ]:
ratios = [w/h for w,h in sizes]

plt.figure(figsize=(8,5))
sns.histplot(ratios, bins=30)

plt.title("Aspect Ratio Distribution")
plt.xlabel("Width / Height")
plt.show()

### Color Distribution Analysis in Artwork Images

### Image Brightness Analysis

### Emotion Estimation Based on Image Brightness

### Distribution of Estimated Emotions (Joy, Calm, Sadness)

In [ ]:
import seaborn as sns

plt.figure(figsize=(6,4))
sns.countplot(x="Emotion", data=emotion_df)

plt.title("Estimated Emotion Distribution")
plt.show()

## Random Sample Images from the Dataset

In [ ]:
import random

plt.figure(figsize=(10,10))

for i in range(9):

    img_name = random.choice(artists)
    img_path = os.path.join(resized_path, img_name)

    img = Image.open(img_path)

    plt.subplot(3,3,i+1)
    plt.imshow(img)
    plt.axis("off")

plt.show()

## Total Number of Images in the Dataset

In [ ]:
print("Total number of images:", len(artists))

## Key Insights from EDA

- The dataset contains more than 8000 artwork images from multiple artists.
- Artists include famous painters such as Van Gogh, Monet, and Leonardo da Vinci.
- Image sizes vary significantly, which requires resizing before training the generative model.
- Color distribution and brightness analysis suggest variations that may correspond to different emotional tones in the artworks.
- Estimated emotion labels (Joy, Calm, Sadness) were generated using brightness heuristics to support emotion-aware generation.
- The dataset provides diverse artistic styles suitable for training a multi-artist generative model.

# Part 2 - Feature Engineering

We now engineer features from three sources:
1. Image filenames - artist name, artwork ID
2. artists.csv - genre, nationality, birth/death year
3. Image pixels - brightness, contrast, saturation, RGB stats, aspect ratio

All categorical features are encoded and all numerical features are normalized.

## FE Step 1 - Import Additional Libraries

In [ ]:
import re
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

print("Feature engineering libraries ready.")

## FE Step 2 - Load artists.csv

In [ ]:
csv_path = os.path.join(path, "artists.csv")

artists_meta = pd.read_csv(csv_path)
print("Shape:", artists_meta.shape)
print("Columns:", artists_meta.columns.tolist())
artists_meta.head()

In [ ]:
print("Missing values:")
print(artists_meta.isnull().sum())

## FE Step 3 - Build Image-Level DataFrame from Filenames

Each filename follows the pattern: ArtistName_ID.jpg

Example: Vincent_van_Gogh_511.jpg

We split each filename to extract the artist name and the artwork index number.

In [ ]:
image_files = [f for f in os.listdir(resized_path) if f.endswith(".jpg")]
print("Total images:", len(image_files))

records = []
for fname in image_files:
    parts = fname.replace(".jpg", "").rsplit("_", 1)
    if len(parts) == 2 and parts[1].isdigit():
        artist_raw = parts[0]
        artwork_id = int(parts[1])
    else:
        artist_raw = fname.replace(".jpg", "")
        artwork_id = -1
    records.append({
        "filename"   : fname,
        "filepath"   : os.path.join(resized_path, fname),
        "artist_raw" : artist_raw,
        "artwork_id" : artwork_id,
    })

df_fe = pd.DataFrame(records)
print("DataFrame shape:", df_fe.shape)
df_fe.head()

## FE Step 4 - Extract Name Parts from Filename

In [ ]:
def parse_artist_name(raw_name):
    parts      = raw_name.split("_")
    first_name = parts[0] if len(parts) >= 1 else ""
    last_name  = parts[-1] if len(parts) >= 2 else ""
    full_name  = " ".join(parts)
    return first_name, last_name, full_name

df_fe[["first_name", "last_name", "artist_name"]] = df_fe["artist_raw"].apply(
    lambda x: pd.Series(parse_artist_name(x))
)

df_fe["name_word_count"] = df_fe["artist_raw"].apply(lambda x: len(x.split("_")))

df_fe[["filename", "artist_name", "first_name", "last_name", "artwork_id", "name_word_count"]].head(8)

## FE Step 5 - Merge with artists.csv Metadata

The CSV provides: genre, nationality, years active, and total paintings per artist.

We normalize the name column to use underscores so it matches the filename format, then merge on that key.

In [ ]:
artists_meta["artist_raw"] = artists_meta["name"].str.replace(" ", "_")

meta_cols    = ["artist_raw", "years", "genre", "nationality", "paintings"]
artists_slim = artists_meta[meta_cols].copy()

df_fe = df_fe.merge(artists_slim, on="artist_raw", how="left")

print("Shape after merge:", df_fe.shape)
df_fe.head(3)

## FE Step 6 - Parse birth_year, death_year, age_at_death, is_alive

The years column contains strings like '1853 - 1890'.

We use a regular expression to extract the four-digit numbers.

In [ ]:
def parse_years(years_str):
    if pd.isna(years_str):
        return pd.Series([None, None, None, 0])
    numbers      = re.findall(r"\d{4}", str(years_str))
    birth_year   = int(numbers[0]) if len(numbers) >= 1 else None
    death_year   = int(numbers[1]) if len(numbers) >= 2 else None
    age_at_death = (death_year - birth_year) if (birth_year and death_year) else None
    is_alive     = 1 if death_year is None else 0
    return pd.Series([birth_year, death_year, age_at_death, is_alive])

df_fe[["birth_year", "death_year", "age_at_death", "is_alive"]] = df_fe["years"].apply(parse_years)

df_fe[["artist_name", "years", "birth_year", "death_year", "age_at_death", "is_alive"]]\
    .drop_duplicates("artist_name").head(8)

## FE Step 7 - Art Era Feature

We classify each artist into a historical art movement era based on their birth year.

| Era | Birth Year Range |
|-----|------------------|
| Medieval | Before 1400 |
| Renaissance | 1400 to 1600 |
| Baroque | 1600 to 1750 |
| Romanticism | 1750 to 1850 |
| Impressionism | 1850 to 1900 |
| Modern | 1900 and later |

In [ ]:
def get_art_era(birth_year):
    if pd.isna(birth_year):
        return "Unknown"
    y = int(birth_year)
    if y < 1400:  return "Medieval"
    elif y < 1600: return "Renaissance"
    elif y < 1750: return "Baroque"
    elif y < 1850: return "Romanticism"
    elif y < 1900: return "Impressionism"
    else:          return "Modern"

df_fe["art_era"] = df_fe["birth_year"].apply(get_art_era)

print("Art era distribution:")
print(df_fe["art_era"].value_counts())

## FE Step 8 - Image Pixel Feature Extraction

We open each image and compute the following pixel-level statistics:

| Feature | Description |
|---------|-------------|
| img_width, img_height | Dimensions in pixels |
| aspect_ratio | width divided by height |
| mean_r, mean_g, mean_b | Average value per RGB channel (0-255) |
| std_r, std_g, std_b | Standard deviation per RGB channel |
| brightness | Mean of the grayscale-converted image |
| contrast | Standard deviation of the grayscale image |
| saturation | Mean of the S channel in HSV color space |
| hue_mean | Mean of the H channel in HSV color space |
| colorfulness | Hasler and Suesstrunk (2003) colorfulness metric |

Note: We use a sample of 500 images for speed. Remove .sample(500) to run on all images.

In [ ]:
def extract_image_features(filepath):
    try:
        img_rgb = Image.open(filepath).convert("RGB")
        img_arr = np.array(img_rgb, dtype=np.float32)
        h, w, _ = img_arr.shape

        r, g, b   = img_arr[:,:,0], img_arr[:,:,1], img_arr[:,:,2]
        gray      = 0.299*r + 0.587*g + 0.114*b
        img_hsv   = np.array(Image.open(filepath).convert("HSV"), dtype=np.float32)

        rg           = r - g
        yb           = 0.5*(r + g) - b
        colorfulness = (np.sqrt(rg.std()**2 + yb.std()**2)
                        + 0.3 * np.sqrt(rg.mean()**2 + yb.mean()**2))

        return {
            "img_width"   : w,
            "img_height"  : h,
            "aspect_ratio": round(w/h, 4),
            "mean_r"      : round(r.mean(), 3),
            "mean_g"      : round(g.mean(), 3),
            "mean_b"      : round(b.mean(), 3),
            "std_r"       : round(r.std(), 3),
            "std_g"       : round(g.std(), 3),
            "std_b"       : round(b.std(), 3),
            "brightness"  : round(gray.mean(), 3),
            "contrast"    : round(gray.std(), 3),
            "saturation"  : round(img_hsv[:,:,1].mean(), 3),
            "hue_mean"    : round(img_hsv[:,:,0].mean(), 3),
            "colorfulness": round(colorfulness, 3),
        }
    except Exception:
        return None


print("Extracting image features from a sample of 500 images...")
sample_df = df_fe.sample(500, random_state=42).copy()

feature_list = []
for idx, row in sample_df.iterrows():
    feats = extract_image_features(row["filepath"])
    if feats:
        feats["filename"] = row["filename"]
        feature_list.append(feats)

img_features_df = pd.DataFrame(feature_list)
print("Done. Extracted features for", len(img_features_df), "images.")
img_features_df.head()

## FE Step 9 - Merge Image Features into Main DataFrame

In [ ]:
df_full = sample_df.merge(img_features_df, on="filename", how="inner")
print("Shape after merging image features:", df_full.shape)
df_full.head(3)

## FE Step 10 - Derived Feature Engineering

We create new features by applying logic on top of the raw extracted values.

| Feature | Logic |
|---------|-------|
| emotion_label | Brightness >= 170 = Joy, >= 100 = Calm, below 100 = Sadness |
| brightness_cat | Bins: Dark / Medium / Bright |
| color_temp | Warm if mean_r > mean_b, otherwise Cool |
| is_portrait | 1 if height is greater than width |
| is_colorful | 1 if colorfulness is above the dataset median |
| dominant_channel | R, G, or B - whichever channel has the highest mean |
| active_years | death_year minus birth_year |
| productivity | Total paintings divided by active years |

In [ ]:
def get_emotion(b):
    if b >= 170:  return "Joy"
    elif b >= 100: return "Calm"
    else:          return "Sadness"

def dominant_ch(r, g, b):
    if r >= g and r >= b:  return "Red"
    elif g >= r and g >= b: return "Green"
    else:                   return "Blue"

df_full["emotion_label"]  = df_full["brightness"].apply(get_emotion)

df_full["brightness_cat"] = pd.cut(
    df_full["brightness"], bins=[0, 85, 170, 255],
    labels=["Dark", "Medium", "Bright"]
)

df_full["color_temp"] = df_full.apply(
    lambda row: "Warm" if row["mean_r"] > row["mean_b"] else "Cool", axis=1
)

df_full["is_portrait"]  = (df_full["img_height"] > df_full["img_width"]).astype(int)
df_full["is_colorful"]  = (df_full["colorfulness"] > df_full["colorfulness"].median()).astype(int)

df_full["dominant_channel"] = df_full.apply(
    lambda row: dominant_ch(row["mean_r"], row["mean_g"], row["mean_b"]), axis=1
)

df_full["active_years"] = df_full["death_year"] - df_full["birth_year"]

df_full["productivity"] = df_full.apply(
    lambda row: round(row["paintings"] / row["active_years"], 3)
    if pd.notna(row["active_years"]) and row["active_years"] > 0 else None, axis=1
)

print("Derived features added.")
df_full[["filename", "emotion_label", "brightness_cat", "color_temp",
         "is_portrait", "is_colorful", "dominant_channel", "productivity"]].head(8)

## FE Step 11 - Encode Categorical Features

Machine learning models require numerical input.

- Label Encoding is used for high-cardinality columns (artist_name, nationality, art_era, dominant_channel). Each unique string value becomes a unique integer.
- One-Hot Encoding is used for low-cardinality columns (emotion_label, color_temp, brightness_cat). Each category becomes its own binary column.

In [ ]:
le = LabelEncoder()

label_encode_cols = ["artist_name", "nationality", "art_era", "dominant_channel"]
for col in label_encode_cols:
    if col in df_full.columns:
        df_full[f"{col}_enc"] = le.fit_transform(df_full[col].astype(str))

print("Label Encoding applied to:", label_encode_cols)

onehot_cols = ["emotion_label", "color_temp", "brightness_cat"]
df_full = pd.get_dummies(df_full, columns=onehot_cols, drop_first=False)

print("One-Hot Encoding applied to:", onehot_cols)
print("New shape:", df_full.shape)

## FE Step 12 - Normalize Numerical Features

MinMaxScaler rescales all numerical features to the range [0, 1].

This prevents features with large value ranges (e.g. mean_r: 0-255) from dominating features with small ranges (e.g. aspect_ratio: 0-3) during model training.

In [ ]:
num_cols = ["brightness", "contrast", "saturation", "colorfulness",
            "mean_r", "mean_g", "mean_b", "std_r", "std_g", "std_b",
            "aspect_ratio", "hue_mean"]

num_cols_present = [c for c in num_cols if c in df_full.columns]

scaler = MinMaxScaler()
df_full[[f"{c}_norm" for c in num_cols_present]] = scaler.fit_transform(df_full[num_cols_present])

print("Normalization complete.")
df_full[[f"{c}_norm" for c in num_cols_present]].describe().round(3)

## FE Step 13 - Visualize Engineered Features

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Feature Engineering - Visual Overview", fontsize=16, fontweight="bold")

# Emotion distribution
emotion_cols = [c for c in df_full.columns if c.startswith("emotion_label_")]
if emotion_cols:
    ec = df_full[emotion_cols].sum()
    ec.index = [c.replace("emotion_label_", "") for c in ec.index]
    ec.plot(kind="bar", ax=axes[0,0], color=["gold", "steelblue", "mediumpurple"])
    axes[0,0].set_title("Emotion Label Distribution")
    axes[0,0].tick_params(axis="x", rotation=0)

# Brightness histogram
axes[0,1].hist(df_full["brightness"], bins=30, color="coral", edgecolor="black")
axes[0,1].axvline(df_full["brightness"].mean(), color="black", linestyle="--", label="Mean")
axes[0,1].set_title("Brightness Distribution")
axes[0,1].set_xlabel("Brightness (0-255)")
axes[0,1].legend()

# Colorfulness histogram
axes[0,2].hist(df_full["colorfulness"], bins=30, color="mediumseagreen", edgecolor="black")
axes[0,2].set_title("Colorfulness Distribution")
axes[0,2].set_xlabel("Colorfulness Score")

# Color temperature pie
ct_cols = [c for c in df_full.columns if c.startswith("color_temp_")]
if ct_cols:
    ct = df_full[ct_cols].sum()
    ct.index = [c.replace("color_temp_", "") for c in ct.index]
    ct.plot(kind="pie", ax=axes[1,0], autopct="%1.1f%%", colors=["tomato", "cornflowerblue"])
    axes[1,0].set_title("Color Temperature")
    axes[1,0].set_ylabel("")

# Art era bar
df_full["art_era"].value_counts().plot(kind="barh", ax=axes[1,1], color="mediumpurple")
axes[1,1].set_title("Art Era Distribution")
axes[1,1].set_xlabel("Count")

# Brightness vs Colorfulness scatter
axes[1,2].scatter(df_full["brightness"], df_full["colorfulness"], alpha=0.4, color="darkorange", s=10)
axes[1,2].set_title("Brightness vs Colorfulness")
axes[1,2].set_xlabel("Brightness")
axes[1,2].set_ylabel("Colorfulness")

plt.tight_layout()
plt.show()

## FE Step 14 - Feature Correlation Heatmap

In [ ]:
corr_features = ["brightness", "contrast", "saturation", "colorfulness",
                 "mean_r", "mean_g", "mean_b", "aspect_ratio",
                 "is_portrait", "is_colorful"]

corr_present = [c for c in corr_features if c in df_full.columns]
corr_matrix  = df_full[corr_present].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, square=True, linewidths=0.5)
plt.title("Feature Correlation Heatmap", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## FE Step 15 - Save Final Feature DataFrame

In [ ]:
drop_cols = [c for c in ["filepath", "artist_raw", "years"] if c in df_full.columns]
df_final  = df_full.drop(columns=drop_cols)

output_path = "/content/artworks_features.csv"
df_final.to_csv(output_path, index=False)

print("Saved. Shape:", df_final.shape)
print("File:", output_path)
print("\nAll columns:")
print(df_final.columns.tolist())

## Feature Engineering Summary

| Feature | Source | Type |
|---------|--------|------|
| artist_name | Filename | Categorical |
| first_name, last_name | Filename | Categorical |
| artwork_id | Filename | Numerical |
| name_word_count | Filename | Numerical |
| genre | artists.csv | Categorical |
| nationality | artists.csv | Categorical |
| paintings | artists.csv | Numerical |
| birth_year, death_year | Parsed from years | Numerical |
| age_at_death | Derived | Numerical |
| is_alive | Derived | Binary |
| active_years | Derived | Numerical |
| art_era | Derived from birth_year | Categorical |
| productivity | paintings / active_years | Numerical |
| img_width, img_height | Pixel | Numerical |
| aspect_ratio | Pixel | Numerical |
| mean_r, mean_g, mean_b | Pixel - RGB | Numerical |
| std_r, std_g, std_b | Pixel - RGB | Numerical |
| brightness | Pixel - Grayscale | Numerical |
| contrast | Pixel - Grayscale | Numerical |
| saturation, hue_mean | Pixel - HSV | Numerical |
| colorfulness | Pixel - Derived formula | Numerical |
| emotion_label | Derived from brightness | Categorical - OHE |
| brightness_cat | Derived from brightness | Categorical - OHE |
| color_temp | Derived from RGB | Categorical - OHE |
| is_portrait | Derived from dimensions | Binary |
| is_colorful | Derived from colorfulness | Binary |
| dominant_channel | Derived from RGB | Categorical - Label Encoded |
| *_enc columns | Label Encoded versions | Numerical |
| *_norm columns | MinMax Normalized versions | Numerical 0 to 1 |